# Keyword Extraction

Extract unique lemmatized words (with POS type) from Original, AI-Curated, and Human-Curated phrases using spaCy.

### Input
- `figures/term_mapping.csv` — produced by PhraseAnalysis.ipynb Step 6

### Output
- `figures/unique_words.csv` — 6 columns: (OriginalWord, OriginalPOS), (AIcuratedWord, AIcuratedPOS), (HumanCuratedWord, HumanCuratedPOS)

In [31]:
import pandas as pd
import spacy

nlp = spacy.load('en_core_web_sm')
df = pd.read_csv('figures/term_mapping.csv')

def extract_unique_words(phrases):
    """Extract unique (lemma, POS) pairs from a list of phrases, removing stopwords and punctuation."""
    word_pos = set()
    for phrase in phrases:
        if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
            continue
        doc = nlp(phrase)
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            lemma = token.lemma_.lower()
            if lemma:
                word_pos.add((lemma, token.pos_))
    return sorted(word_pos)

# Extract unique words for each type
cols = {'original': 'Original', 'AIcurated': 'AIcurated', 'HumanCurated': 'HumanCurated'}
word_lists = {}
for col, label in cols.items():
    phrases = df[col][df[col] != ''].unique()
    wp = extract_unique_words(phrases)
    word_lists[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique words')

# Build output DataFrame — 3 independent column-pairs, padded
max_len = max(len(v) for v in word_lists.values())
frames = []
for label in cols.values():
    wp = word_lists[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_out = pd.concat(frames, axis=1)

csv_path = 'figures/unique_words.csv'
df_out.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_out.head(15)

Original: 1582 phrases → 878 unique words
AIcurated: 876 phrases → 554 unique words
HumanCurated: 402 phrases → 375 unique words

Saved figures/unique_words.csv  (878 rows)


,OriginalWord,OriginalPOS,AIcuratedWord,AIcuratedPOS,HumanCuratedWord,HumanCuratedPOS
0,1,NUM,2d,PROPN,2d/3d,NUM
1,1/3,NUM,3d,ADJ,abstract,ADJ
2,15,NUM,3d,NOUN,alignment,NOUN
3,2,NUM,abstract,ADJ,ambiguous,ADJ
4,2d,PROPN,add,VERB,analysis,NOUN
5,3,NUM,additional,ADJ,analyze,VERB
6,3d,ADJ,align,VERB,annotation,NOUN
7,3d,NOUN,alignment,NOUN,appeal,VERB
8,4,NUM,ambiguous,ADJ,arbitrary,ADJ
9,5,NUM,analysis,NOUN,arrangement,NOUN


In [25]:
# ── Helper functions ────────────────────────────────────────────────────────────
from collections import Counter

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

def extract_lemmas_from_phrase(phrase):
    """Extract set of lemmatized words from a single phrase."""
    if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
        return set()
    doc = nlp(phrase)
    return {token.lemma_.lower() for token in doc
            if not token.is_stop and not token.is_punct and not token.is_space and token.lemma_.lower()}

def extract_unique_lemmas(phrases):
    """Extract sorted unique lemmatized words from a list of phrases."""
    words = set()
    for phrase in phrases:
        words |= extract_lemmas_from_phrase(phrase)
    return sorted(words)

print('Helper functions defined.')

Helper functions defined.


In [11]:
# ── Load paired image data (sheet 1103808983) ─────────────────────────────────
import json

spreadsheet_id = '1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc'
sheet_id = '1103808983'
df_pairs = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet_id}&format=csv'
)
print(f'Shape: {df_pairs.shape}')
print(f'Columns: {df_pairs.columns.tolist()}')

# Filter out rows with ToRemove
before = len(df_pairs)
df_pairs = df_pairs[df_pairs['ToRemove'].isna() | (df_pairs['ToRemove'].astype(str).str.strip() == '')]
print(f'Rows after removing ToRemove: {len(df_pairs)} (removed {before - len(df_pairs)})')
df_pairs.head(3)

Shape: (698, 33)
Columns: ['ToRemove', 'index', 'moreComplexImageName', 'lessComplexImageName', 'moreComplexImageChosen', 'lessComplexImageChosen', 'VisTypesMoreComplex', 'MASSVISTypeMoreComplexImageType', 'MASSVISTypeLessComplexImageType', 'VisTypesMoreComplexImageType', 'VisTypesLessComplexImageType', 'commentCollected', 'CommentCollectedOnMoreComplexFig', 'RowInOrgSheets', 'OrderSwitched', 'ExtractedPhrasesMoreComplex', '(to merge with next col) CuratePhrasesMoreComplex', 'CurateCognitiveMoreComplex', 'CuratePhrasesMoreComplex', '(Final.MappedBack)HumanCuratedFinalKeywordsMoreComplex', 'CommentCollectedOnLessComplexFig', 'ExtractedPhrasesLessComplex', '(to merge with next col) CuratePhrasesLessComplex', 'CurateCognitiveLessComplex', 'CuratePhrasesLessComplex', '(Final.MappedBack)HumanCuratedFinalKeywordsLessComplex', '(Not used - old prompt single words many repetitions) ExtractedKeywordsLessComplexCurated', '(Not used) ExtractedKeywordsMoreComplex', '(Not used) ExtractKeywordsMoreC

,ToRemove,index,moreComplexImageName,lessComplexImageName,moreComplexImageChosen,lessComplexImageChosen,VisTypesMoreComplex,MASSVISTypeMoreComplexImageType,MASSVISTypeLessComplexImageType,VisTypesMoreComplexImageType,...,CurateCognitiveLessComplex,CuratePhrasesLessComplex,(Final.MappedBack)HumanCuratedFinalKeywordsLessComplex,(Not used - old prompt single words many repetitions) ExtractedKeywordsLessComplexCurated,(Not used) ExtractedKeywordsMoreComplex,(Not used) ExtractKeywordsMoreComplexCurated,(Not used--old single word) ExtractedKeywordsMoreComplexCurated,(Not used--old single word) ExtractedKeywordsLessComplex,(Not used) ExtractedCognitiveMoreComplexCurated,(Not used - old prompt single words many repetitions) ExtractedCognitiveLessComplexCurated
0,NaN,344,InfoVisC.115.4.png,InfoVisC.133.5(3).png,NaN,NaN,NaN,Area,Diagram,Area,...,NaN,NaN,NaN,NaN,"{\n ""Density / Clutter"": [\n ""number"",\n ...","{\n ""Density / Clutter"": [\n ""number of th...","{\n ""Density / Clutter"": [\n ""number of th...",NaN,"{\n ""Immediacy / Cognitive Load"": [\n ""dif...",NaN
1,NaN,176,InfoVisC.173.7(1).png,VASTJ.1139.3(2).png,NaN,NaN,NaN,Area,Trees and Networks,Area,...,"{\n ""Data Density / Image Clutter"": [\n ""l...","{\n ""Data Density / Image Clutter"": [\n ""l...","{\n ""Data Density / Image Clutter"": [\n ""l...","{\n ""Density / Clutter"": [\n ""less informa...","{\n ""Density / Clutter"": [\n ""where i'm su...",NaN,"{\n ""Others"": [\n ""where to look""\n ]\n}","{\n ""Density / Clutter"": [\n ""less informa...","{\n ""Immediacy / Cognitive Load"": [\n ""whe...","{\n ""Immediacy / Cognitive Load"": [\n ""eas..."
2,NaN,521,InfoVisJ.1634.1.png,SciVisJ.728.14.png,NaN,NaN,NaN,Area,Table,Area,...,"{\n ""Semantics / Text Legibility"": [\n ""ex...","{\n ""Semantics / Text Legibility"": [\n ""ex...","{\n ""Semantics / Text Legibility"": [\n ""ex...","{\n ""Semantics / Text Legibility"": [\n ""le...","{\n ""Semantics / Text Legibility"": [\n ""no...","{\n ""Semantics / Text Legibility"": [\n ""no...","{\n ""Semantics / Text Legibility"": [\n ""no...","{\n ""Semantics / Text Legibility"": [\n ""le...","{\n ""Immediacy / Cognitive Load"": [\n ""no ...",NaN


In [18]:
# ── Build (topic, original) → (AIcurated, HumanCurated) mapping from sheet 1166121277 ──
sheet_map_id = '1166121277'
df_map_src = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?gid={sheet_map_id}&format=csv'
)

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

# Build topic-aware mapping: (topic, original) → (AIcurated, HumanCurated)
topic_orig_to_curated = {}
for topic in topic_cols:
    col_orig = topic
    col_ai   = f'{topic} (AIcurated)'
    col_hc   = f'{topic} (HumanCuration)'
    for _, row in df_map_src.iterrows():
        orig = row.get(col_orig)
        if not isinstance(orig, str) or orig.strip() == '':
            continue
        orig = orig.strip()
        key = (topic, orig)
        if key not in topic_orig_to_curated:
            ai = str(row.get(col_ai, '')).strip() if isinstance(row.get(col_ai), str) else ''
            hc = str(row.get(col_hc, '')).strip() if isinstance(row.get(col_hc), str) else ''
            topic_orig_to_curated[key] = (ai, hc)

print(f'Mapping entries: {len(topic_orig_to_curated)}')
for k in list(topic_orig_to_curated.keys())[:5]:
    print(f'  ({k[0]}, "{k[1]}") → AI: "{topic_orig_to_curated[k][0]}", HC: "{topic_orig_to_curated[k][1]}"')

Mapping entries: 1601
  (Immediacy / Cognitive Load, "need of focus") → AI: "need focus", HC: "attention/squinting to understand"
  (Immediacy / Cognitive Load, "squint my eyes to understand") → AI: "require squinting to understand", HC: "attention/squinting to understand"
  (Immediacy / Cognitive Load, "requires concentration") → AI: "requires concentration", HC: "attention/squinting to understand"
  (Immediacy / Cognitive Load, "complicated at first glance") → AI: "complicated", HC: "complicated to process"
  (Immediacy / Cognitive Load, "complicated to look at") → AI: "complicated", HC: "complicated to process"


In [19]:
# ── Image-based phrase matching by VisType ─────────────────────────────────────
import json

CANONICAL_VISTYPES = {'Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text'}

# Collect per-image records: (image_name, VisType, topic, original) → deduplicated
seen = set()
records = []
unmatched_phrases = set()

for idx, row in df_pairs.iterrows():
    # Process both sides: MoreComplex and LessComplex
    for img_col, vt_col, json_col_idx in [
        ('moreComplexImageName', 'VisTypesMoreComplexImageType', 18),
        ('lessComplexImageName', 'VisTypesLessComplexImageType', 24),
    ]:
        img_name = row.get(img_col)
        raw_vt = row.get(vt_col)
        if not isinstance(img_name, str) or not isinstance(raw_vt, str):
            continue
        raw_vt = raw_vt.strip()

        # Only keep exact canonical VisTypes
        if raw_vt not in CANONICAL_VISTYPES:
            continue

        # Parse JSON phrases
        json_str = df_pairs.iloc[df_pairs.index.get_loc(idx), json_col_idx]
        if not isinstance(json_str, str) or json_str.strip() == '':
            continue
        try:
            phrase_dict = json.loads(json_str)
        except json.JSONDecodeError:
            continue

        for topic, phrases_list in phrase_dict.items():
            if not isinstance(phrases_list, list):
                continue
            for orig_phrase in phrases_list:
                if not isinstance(orig_phrase, str) or orig_phrase.strip() == '':
                    continue
                orig_phrase = orig_phrase.strip()

                # Deduplicate: same image may appear in multiple rows
                dedup_key = (img_name, raw_vt, topic, orig_phrase)
                if dedup_key in seen:
                    continue
                seen.add(dedup_key)

                ai, hc = topic_orig_to_curated.get((topic, orig_phrase), ('', ''))
                if not ai and not hc:
                    unmatched_phrases.add(orig_phrase)
                records.append({
                    'ImageName': img_name,
                    'VisType': raw_vt,
                    'Topic': topic,
                    'Original': orig_phrase,
                    'AIcurated': ai,
                    'HumanCurated': hc,
                })

df_image_phrases = pd.DataFrame(records)
csv_path = 'figures/image_phrase_matching.csv'
df_image_phrases.to_csv(csv_path, index=False)
n_images = df_image_phrases['ImageName'].nunique()
print(f'Records: {len(df_image_phrases)} ({n_images} unique images)')
print(f'Unique VisTypes: {sorted(df_image_phrases["VisType"].unique())}')
print(f'Unmatched phrases: {len(unmatched_phrases)}')
if unmatched_phrases:
    for p in sorted(unmatched_phrases):
        print(f'  - "{p}"')
print(f'\nSaved {csv_path}')
df_image_phrases.head(10)

Records: 1909 (493 unique images)
Unique VisTypes: ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']
Unmatched phrases: 3
  - "alignment"
  - "make sense."
  - "set of geometry"

Saved figures/image_phrase_matching.csv


,ImageName,VisType,Topic,Original,AIcurated,HumanCurated
0,InfoVisC.115.4.png,Area,Data Density / Image Clutter,number of things in the same area,many things in one area,many points/lines/shapes/elements
1,InfoVisC.115.4.png,Area,"Color, Symbol, and Texture Details",colors in the same area,colors clustered together,color variety/arrangement/distribution
2,InfoVisC.115.4.png,Area,Immediacy / Cognitive Load,dificult to see whats about,hard to see,hard to see/tell/visualize
3,InfoVisC.173.7(1).png,Area,Immediacy / Cognitive Load,don't understand where to look.,don't know where to look,unclear where to look/what to see
4,VASTJ.1139.3(2).png,Node-link,Data Density / Image Clutter,less information (elements),less information,little data/info
5,VASTJ.1139.3(2).png,Node-link,Immediacy / Cognitive Load,less information to process,reduces the effort to comprehend,easy to interpret/read/understand
6,InfoVisJ.1634.1.png,Area,Semantics / Text Legibility,no description,no description,lack of/not enough axis labels/legend/annotati...
7,InfoVisJ.1634.1.png,Area,Immediacy / Cognitive Load,no idea what this is,no idea,unclear meaning/confusing
8,SciVisJ.728.14.png,Grid,Semantics / Text Legibility,explanations,explanation text,explanation text
9,SciVisJ.728.14.png,Grid,Semantics / Text Legibility,legend,legend,too much legend


In [29]:
# ── Per-topic unique words with image counts (Original, AIcurated, HumanCurated) ─
frames = []
for topic in topic_cols:
    subset = df_image_phrases[df_image_phrases['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per lemma
        lemma_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_lemmas = set()
            for phrase in phrases:
                img_lemmas |= extract_lemmas_from_phrase(phrase)
            for lemma in img_lemmas:
                lemma_images[lemma] += 1

        sorted_items = sorted(lemma_images.items())
        col_data[f'{topic}\n{label}'] = [w for w, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    # Pad to same length within this topic
    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data[f'{topic}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{topic}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{topic}\nHumanCurated'] if x])
    print(f'{topic}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics = pd.concat(frames, axis=1)

csv_path = 'figures/unique_words_by_topic.csv'
df_topics.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics)} rows, {len(df_topics.columns)} columns)')
df_topics.head(10)

Immediacy / Cognitive Load: Orig=218, AI=96, HC=66
Data Density / Image Clutter: Orig=179, AI=86, HC=55
Visual Encoding Clarity: Orig=254, AI=147, HC=137
Semantics / Text Legibility: Orig=162, AI=115, HC=79
Schema: Orig=172, AI=93, HC=26
Color, Symbol, and Texture Details: Orig=186, AI=133, HC=86
Aesthetics Uncertainty: Orig=74, AI=51, HC=28

Saved figures/unique_words_by_topic.csv  (254 rows, 42 columns)


,Immediacy / Cognitive Load\nOriginal,Immediacy / Cognitive Load\nOriginal Count,Immediacy / Cognitive Load\nAIcurated,Immediacy / Cognitive Load\nAIcurated Count,Immediacy / Cognitive Load\nHumanCurated,Immediacy / Cognitive Load\nHumanCurated Count,Data Density / Image Clutter\nOriginal,Data Density / Image Clutter\nOriginal Count,Data Density / Image Clutter\nAIcurated,Data Density / Image Clutter\nAIcurated Count,...,"Color, Symbol, and Texture Details\nAIcurated","Color, Symbol, and Texture Details\nAIcurated Count","Color, Symbol, and Texture Details\nHumanCurated","Color, Symbol, and Texture Details\nHumanCurated Count",Aesthetics Uncertainty\nOriginal,Aesthetics Uncertainty\nOriginal Count,Aesthetics Uncertainty\nAIcurated,Aesthetics Uncertainty\nAIcurated Count,Aesthetics Uncertainty\nHumanCurated,Aesthetics Uncertainty\nHumanCurated Count
0,1,1,analysis,1,analysis,3,1/3,1,analyze,1,...,ambiguous,5,ambiguous,5,appealing,2,appeal,4,appeal,5
1,15,1,ascertain,2,analyze,5,2,2,area,4,...,appeal,2,appeal,2,arbitrary,1,appealing,1,arbitrary,1
2,2,1,careful,1,attention,3,4,1,box,2,...,arrangement,1,arrangement,85,attractive,1,arbitrary,1,attractive,3
3,5,1,challenge,1,clear,1,abundance,1,busy,7,...,background,4,background,5,bad,3,attractive,1,clarity,5
4,able,1,clear,1,compare,11,ammount,1,chart,3,...,bar,1,bar,2,black,1,bad,3,color,8
5,accessible,1,clearly,3,complicated,8,analyse,1,circle,4,...,black,9,black,14,blurry,1,blurry,1,composition,1
6,accurate,1,clue,1,confuse,24,annotations,1,close,2,...,blend,1,blend,1,bore,1,boring,1,confuse,19
7,add,1,compare,2,datum,7,area,4,cluster,2,...,blue,5,blue,2,calligraphy,1,calligraphy,1,contrast,8
8,allow,2,complex,4,derive,2,arrow,1,cluttered,7,...,blur,2,blurry,3,chaotic,1,chaotic,1,convolute,1
9,analysis,2,complicated,5,describe,2,aspect,1,color,1,...,blurry,1,bold,1,choice,3,choice,1,distract,19


In [30]:
# ── Unique words by VisType with image counts (Original, AIcurated, HumanCurated)
CANONICAL_VISTYPES = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_image_phrases[df_image_phrases['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per lemma
        lemma_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_lemmas = set()
            for phrase in phrases:
                img_lemmas |= extract_lemmas_from_phrase(phrase)
            for lemma in img_lemmas:
                lemma_images[lemma] += 1

        sorted_items = sorted(lemma_images.items())
        col_data[f'{vt}\n{label}'] = [w for w, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    # Pad to same length within this VisType
    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data[f'{vt}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{vt}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{vt}\nHumanCurated'] if x])
    print(f'{vt}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype = pd.concat(frames, axis=1)

csv_path = 'figures/unique_words_by_vistype.csv'
df_vistype.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype)} rows, {len(df_vistype.columns)} columns)')
df_vistype.head(10)

Area: Orig=259, AI=196, HC=166
Bar: Orig=143, AI=119, HC=123
Cont.-ColorPatn: Orig=179, AI=144, HC=148
Glyph: Orig=203, AI=151, HC=143
Grid: Orig=205, AI=164, HC=166
Line: Orig=164, AI=129, HC=118
Node-link: Orig=235, AI=171, HC=164
Point: Orig=196, AI=151, HC=153
Text: Orig=134, AI=107, HC=119

Saved figures/unique_words_by_vistype.csv  (259 rows, 54 columns)


,Area\nOriginal,Area\nOriginal Count,Area\nAIcurated,Area\nAIcurated Count,Area\nHumanCurated,Area\nHumanCurated Count,Bar\nOriginal,Bar\nOriginal Count,Bar\nAIcurated,Bar\nAIcurated Count,...,Point\nAIcurated,Point\nAIcurated Count,Point\nHumanCurated,Point\nHumanCurated Count,Text\nOriginal,Text\nOriginal Count,Text\nAIcurated,Text\nAIcurated Count,Text\nHumanCurated,Text\nHumanCurated Count
0,1/3,1,abstract,1,2d/3d,1,1,1,2d,1,...,2d,1,2d/3d,2,2,1,ambiguous,1,ambiguous,1
1,3,1,align,1,abstract,1,2,2,ambiguous,1,...,3d,1,abstract,1,application,1,annotation,1,annotation,2
2,abstract,1,ambiguous,2,alignment,1,2d,1,analysis,1,...,add,1,ambiguous,1,arbitrary,1,busy,1,arrangement,7
3,accessible,1,analyze,1,ambiguous,2,alignment,1,axis,2,...,ambiguous,1,analyze,1,ascertain,1,calligraphy,1,attractive,1
4,africa,1,appeal,1,analyze,1,analysis,1,bar,7,...,animal,1,annotation,6,big,1,chart,1,axis,6
5,allow,1,area,2,annotation,5,axis,2,black,1,...,annotation,1,arrangement,10,bit,1,circle,1,bar,1
6,ambiguous,1,ascertain,1,appeal,1,bar,7,blue,1,...,appealing,1,aspect,1,bunch,1,clear,3,biology,4
7,analyse,1,axis,5,arrangement,16,black,1,busy,1,...,area,1,attractive,1,busy,1,clearly,1,chart,3
8,analyze,1,background,1,axis,11,blue,1,change,1,...,aspect,1,axis,14,calligraphy,1,cluttered,2,chemical,4
9,apart,1,black,1,background,1,box,1,chart,3,...,axis,8,bar,1,change,1,code,1,circle,1
